# SCF Phase 2 shard 05

Sweeps: correctness. Jobs: 28. Projected: 5.0 h
(safety-factor 1.8x applied). Grids hash: `68970a975545`.
Code source: github.com/hugogobato/scf-confounding-frontier @ tag `phase2-freeze` (pinned for
reproducibility).
Pre-registration: `docs/phase2_preregistration.md` (thresholds frozen before
any data generation; deviation register D1-D7 included there).

Resume-safe: completed cells are skipped on rerun (checkpoint parquet per
cell). If the notebook approaches the Colab wall limit it finishes the
current cell and stops cleanly; rerun to continue.

In [ ]:
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[_v] = "1"
!pip install -q "numpy>=2.0" "scipy>=1.14" "pandas>=2.2" "pyarrow>=16" scikit-learn

In [ ]:
!git clone --depth 1 --branch phase2-freeze \
    https://github.com/hugogobato/scf-confounding-frontier.git scf_repo
import sys, hashlib, json
sys.path.insert(0, "scf_repo/code")
# verify the pinned code matches the manifest recorded at generation time
EXPECTED = json.loads("{\"de_formulas.py\": \"5dffb441b638\", \"simulator.py\": \"ef31ca2a201b\", \"estimators.py\": \"7e27f25b2330\", \"detection.py\": \"06586fe60b9f\", \"runners.py\": \"df67486b60f5\"}")
for fname, short in EXPECTED.items():
    h = hashlib.sha256(open(f"scf_repo/code/{fname}", "rb").read()).hexdigest()[:12]
    assert h == short, f"code mismatch: {fname} ({h} != {short})"
print("code verified against generation-time hashes")

In [ ]:
import json, time, traceback
from multiprocessing import Pool
from runners import run_cell

JOBS = json.loads("[{\"config\": {\"n\": 2000, \"p\": 4000, \"r\": 5, \"l\": [4.242640687119286, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476], \"theta\": 1.5707963267948966, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": true}, \"config_id\": \"2bf5482ddbb9\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 150, \"raw_path\": \"data/sim/correctness/raw/2bf5482ddbb9.parquet\", \"means_path\": \"data/sim/correctness/means/2bf5482ddbb9.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 8000, \"p\": 1600, \"r\": 1, \"l\": [0.22360679774997896], \"theta\": 0.5235987755982988, \"profile\": \"sub\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"a662a458fbf2\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 150, \"raw_path\": \"data/sim/correctness/raw/a662a458fbf2.parquet\", \"means_path\": \"data/sim/correctness/means/a662a458fbf2.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 8000, \"p\": 1600, \"r\": 5, \"l\": [1.3416407864998738, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"6c306ab4e131\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 150, \"raw_path\": \"data/sim/correctness/raw/6c306ab4e131.parquet\", \"means_path\": \"data/sim/correctness/means/6c306ab4e131.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 8000, \"p\": 1600, \"r\": 1, \"l\": [1.3416407864998738], \"theta\": 0.5235987755982988, \"profile\": \"super\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"f3afd35b7300\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 150, \"raw_path\": \"data/sim/correctness/raw/f3afd35b7300.parquet\", \"means_path\": \"data/sim/correctness/means/f3afd35b7300.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 2000, \"p\": 1600, \"r\": 1, \"l\": [0.4472135954999579], \"theta\": 0.5235987755982988, \"profile\": \"sub\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"f7702516d436\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 200, \"raw_path\": \"data/sim/correctness/raw/f7702516d436.parquet\", \"means_path\": \"data/sim/correctness/means/f7702516d436.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 2000, \"p\": 1600, \"r\": 1, \"l\": [2.6832815729997477], \"theta\": 0.5235987755982988, \"profile\": \"super\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"364a9f6c32e7\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 200, \"raw_path\": \"data/sim/correctness/raw/364a9f6c32e7.parquet\", \"means_path\": \"data/sim/correctness/means/364a9f6c32e7.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 2000, \"p\": 1600, \"r\": 5, \"l\": [0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579], \"theta\": 0.5235987755982988, \"profile\": \"sub\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"fa88eb8654fe\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 200, \"raw_path\": \"data/sim/correctness/raw/fa88eb8654fe.parquet\", \"means_path\": \"data/sim/correctness/means/fa88eb8654fe.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 2000, \"p\": 1600, \"r\": 5, \"l\": [2.6832815729997477, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"2820b3650b7b\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 200, \"raw_path\": \"data/sim/correctness/raw/2820b3650b7b.parquet\", \"means_path\": \"data/sim/correctness/means/2820b3650b7b.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 2000, \"p\": 1600, \"r\": 5, \"l\": [2.6832815729997477, 2.6832815729997477, 2.6832815729997477, 2.6832815729997477, 2.6832815729997477], \"theta\": 0.5235987755982988, \"profile\": \"super\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"1d2f7d065648\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 200, \"raw_path\": \"data/sim/correctness/raw/1d2f7d065648.parquet\", \"means_path\": \"data/sim/correctness/means/1d2f7d065648.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 2000, \"p\": 1600, \"r\": 25, \"l\": [2.6832815729997477, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"a599df9bde9f\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 200, \"raw_path\": \"data/sim/correctness/raw/a599df9bde9f.parquet\", \"means_path\": \"data/sim/correctness/means/a599df9bde9f.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 2000, \"p\": 1600, \"r\": 5, \"l\": [0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579], \"theta\": 0.0, \"profile\": \"sub\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"a57883cc7a3b\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 150, \"raw_path\": \"data/sim/correctness/raw/a57883cc7a3b.parquet\", \"means_path\": \"data/sim/correctness/means/a57883cc7a3b.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 2000, \"p\": 1600, \"r\": 5, \"l\": [2.6832815729997477, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579], \"theta\": 0.0, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"0317f8bb81d0\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 150, \"raw_path\": \"data/sim/correctness/raw/0317f8bb81d0.parquet\", \"means_path\": \"data/sim/correctness/means/0317f8bb81d0.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 2000, \"p\": 1600, \"r\": 5, \"l\": [0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579], \"theta\": 1.5707963267948966, \"profile\": \"sub\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"91e20e9ac686\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 150, \"raw_path\": \"data/sim/correctness/raw/91e20e9ac686.parquet\", \"means_path\": \"data/sim/correctness/means/91e20e9ac686.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 2000, \"p\": 1600, \"r\": 5, \"l\": [2.6832815729997477, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579], \"theta\": 1.5707963267948966, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"9a4aca17665f\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 150, \"raw_path\": \"data/sim/correctness/raw/9a4aca17665f.parquet\", \"means_path\": \"data/sim/correctness/means/9a4aca17665f.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 500, \"p\": 2500, \"r\": 1, \"l\": [1.118033988749895], \"theta\": 0.5235987755982988, \"profile\": \"sub\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": true}, \"config_id\": \"b0fc97bfbd47\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 600, \"raw_path\": \"data/sim/correctness/raw/b0fc97bfbd47.parquet\", \"means_path\": \"data/sim/correctness/means/b0fc97bfbd47.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 500, \"p\": 2500, \"r\": 1, \"l\": [6.708203932499369], \"theta\": 0.5235987755982988, \"profile\": \"super\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": true}, \"config_id\": \"ea5baffc42f7\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 600, \"raw_path\": \"data/sim/correctness/raw/ea5baffc42f7.parquet\", \"means_path\": \"data/sim/correctness/means/ea5baffc42f7.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 500, \"p\": 2500, \"r\": 5, \"l\": [1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895], \"theta\": 0.5235987755982988, \"profile\": \"sub\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": true}, \"config_id\": \"662803b9f9a1\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 600, \"raw_path\": \"data/sim/correctness/raw/662803b9f9a1.parquet\", \"means_path\": \"data/sim/correctness/means/662803b9f9a1.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 500, \"p\": 2500, \"r\": 5, \"l\": [6.708203932499369, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": true}, \"config_id\": \"b2bb511dea4a\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 600, \"raw_path\": \"data/sim/correctness/raw/b2bb511dea4a.parquet\", \"means_path\": \"data/sim/correctness/means/b2bb511dea4a.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 500, \"p\": 2500, \"r\": 5, \"l\": [6.708203932499369, 6.708203932499369, 6.708203932499369, 6.708203932499369, 6.708203932499369], \"theta\": 0.5235987755982988, \"profile\": \"super\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": true}, \"config_id\": \"e3b38b66b852\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 600, \"raw_path\": \"data/sim/correctness/raw/e3b38b66b852.parquet\", \"means_path\": \"data/sim/correctness/means/e3b38b66b852.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 500, \"p\": 2500, \"r\": 25, \"l\": [6.708203932499369, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": true}, \"config_id\": \"3919b2a463c8\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 600, \"raw_path\": \"data/sim/correctness/raw/3919b2a463c8.parquet\", \"means_path\": \"data/sim/correctness/means/3919b2a463c8.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 500, \"p\": 1000, \"r\": 1, \"l\": [0.7071067811865476], \"theta\": 0.5235987755982988, \"profile\": \"sub\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": true}, \"config_id\": \"5b89dceca0cb\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 600, \"raw_path\": \"data/sim/correctness/raw/5b89dceca0cb.parquet\", \"means_path\": \"data/sim/correctness/means/5b89dceca0cb.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 500, \"p\": 1000, \"r\": 1, \"l\": [4.242640687119286], \"theta\": 0.5235987755982988, \"profile\": \"super\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": true}, \"config_id\": \"a482e115ae64\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 600, \"raw_path\": \"data/sim/correctness/raw/a482e115ae64.parquet\", \"means_path\": \"data/sim/correctness/means/a482e115ae64.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 500, \"p\": 1000, \"r\": 5, \"l\": [0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476], \"theta\": 0.5235987755982988, \"profile\": \"sub\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": true}, \"config_id\": \"c29a39d94d81\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 600, \"raw_path\": \"data/sim/correctness/raw/c29a39d94d81.parquet\", \"means_path\": \"data/sim/correctness/means/c29a39d94d81.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 500, \"p\": 1000, \"r\": 5, \"l\": [4.242640687119286, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": true}, \"config_id\": \"9f8e1a122970\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 600, \"raw_path\": \"data/sim/correctness/raw/9f8e1a122970.parquet\", \"means_path\": \"data/sim/correctness/means/9f8e1a122970.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 500, \"p\": 1000, \"r\": 5, \"l\": [4.242640687119286, 4.242640687119286, 4.242640687119286, 4.242640687119286, 4.242640687119286], \"theta\": 0.5235987755982988, \"profile\": \"super\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": true}, \"config_id\": \"2583c6e010a4\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 600, \"raw_path\": \"data/sim/correctness/raw/2583c6e010a4.parquet\", \"means_path\": \"data/sim/correctness/means/2583c6e010a4.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 500, \"p\": 1000, \"r\": 25, \"l\": [4.242640687119286, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": true}, \"config_id\": \"9da42d209cad\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 600, \"raw_path\": \"data/sim/correctness/raw/9da42d209cad.parquet\", \"means_path\": \"data/sim/correctness/means/9da42d209cad.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 2000, \"p\": 1000, \"r\": 1, \"l\": [0.3535533905932738], \"theta\": 0.5235987755982988, \"profile\": \"sub\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"5fa07d17fce1\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 200, \"raw_path\": \"data/sim/correctness/raw/5fa07d17fce1.parquet\", \"means_path\": \"data/sim/correctness/means/5fa07d17fce1.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 2000, \"p\": 1000, \"r\": 1, \"l\": [2.121320343559643], \"theta\": 0.5235987755982988, \"profile\": \"super\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"2b580ba4855f\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 200, \"raw_path\": \"data/sim/correctness/raw/2b580ba4855f.parquet\", \"means_path\": \"data/sim/correctness/means/2b580ba4855f.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}]")

def _safe(job):
    try:
        return run_cell(job)
    except Exception as e:
        print('[FAIL]', job['config_id'], repr(e))
        traceback.print_exc()
        return job['config_id'], -1.0

t0 = time.time()
results = []
for i, job in enumerate(JOBS):
    if time.time() - t0 > 8.6 * 3600:
        print('[WALL LIMIT] stopping cleanly after', i, 'jobs')
        break
    results.append(_safe(job))
print('shard done:', results)

In [ ]:
import hashlib, json, glob, os
manifest = {'shard_id': 5, 'files': {}}
os.makedirs('data', exist_ok=True)
for f in sorted(glob.glob('data/**/*.parquet', recursive=True)) + \
         sorted(glob.glob('data/**/*.npz', recursive=True)):
    h = hashlib.sha256(open(f, 'rb').read()).hexdigest()
    manifest['files'][f] = h
with open('data/manifest.json', 'w') as fh:
    json.dump(manifest, fh, indent=1)
print(json.dumps(manifest['files'], indent=1))

In [ ]:
import shutil
archive = shutil.make_archive('scf_shard_{:02d}'.format(5), 'zip', 'data')
print('archived:', archive)
output_file = archive
try:
    from google.colab import files
    files.download(output_file)
    print('Downloaded:', output_file)
except Exception as e:
    print('(Not on Colab / download skipped):', e)